In [19]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np

In [ ]:
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1
)

detector = vision.HandLandmarker.create_from_options(options)

base_options_pose = python.BaseOptions(model_asset_path='pose_landmarker.task')

pose_options = vision.PoseLandmarkerOptions(
    base_options=base_options_pose
)

pose_detector = vision.PoseLandmarker.create_from_options(pose_options)

prev_elbow_x = 0

In [ ]:
# Open camera
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Convert image
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

    result = detector.detect(mp_image)

    lm_list = []

    if result.hand_landmarks:
        for hand_landmarks in result.hand_landmarks:
            h, w, _ = frame.shape

            for id, lm in enumerate(hand_landmarks):
                cx, cy = int(lm.x * w), int(lm.y * h)
                lm_list.append((id, cx, cy))
                cv2.circle(frame, (cx, cy), 5, (0, 255, 0), -1)

    fingers = []

    if len(lm_list) >= 21:
        # Thumb (x comparison)
        if lm_list[4][1] > lm_list[3][1]:
            fingers.append(1)
        else:
            fingers.append(0)

        # Other fingers (y comparison)
        tip_ids = [8, 12, 16, 20]

        for tip in tip_ids:
            if lm_list[tip][2] < lm_list[tip - 2][2]:
                fingers.append(1)
            else:
                fingers.append(0)

        total_fingers = fingers.count(1)

        cv2.putText(frame, f'Fingers: {total_fingers}', (10, 70),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

    gesture = "None"

    if len(fingers) == 5:
        if fingers == [0, 0, 0, 0, 0]:
            gesture = "STOP"
        elif fingers == [0, 1, 0, 0, 0]:
            gesture = "MOVE"
        elif fingers == [0, 1, 1, 0, 0]:
            gesture = "ROTATE"
        elif fingers == [1, 1, 1, 1, 1]:
            gesture = "OPEN"

    cv2.putText(frame, gesture, (10, 120),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    sweep = "NONE"
    command = "NONE"

    pose_result = pose_detector.detect(mp_image)

    if pose_result.pose_landmarks:
        h, w, _ = frame.shape
        right_elbow = pose_result.pose_landmarks[0][14]
        elbow_x = int(right_elbow.x * w)
        elbow_y = int(right_elbow.y * h)

        cv2.circle(frame, (elbow_x, elbow_y), 8, (0, 0, 255), -1)

        if prev_elbow_x != 0:
            diff = elbow_x - prev_elbow_x
            if diff > 25:
                sweep = "RIGHT"
            elif diff < -25:
                sweep = "LEFT"

        prev_elbow_x = elbow_x
    else:
        prev_elbow_x = 0

    if gesture == "MOVE" and sweep == "LEFT":
        command = "MOVE_LEFT"
    elif gesture == "MOVE" and sweep == "RIGHT":
        command = "MOVE_RIGHT"
    elif gesture == "STOP":
        command = "STOP"

    cv2.putText(frame, f'Sweep: {sweep}', (10, 170),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
    cv2.putText(frame, f'Command: {command}', (10, 220),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2)

    cv2.imshow("Hand Tracking", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()